# STA 556 — Homework 4: Simulation & Resampling

**Name:** Benjamin Owusu Acquah  
**Seed:** 556 (single explicit `np.random.default_rng(556)` object used for every random step)

Order matters: the RNG stream is consumed top to bottom, so run all cells in order (Restart kernel → Run All).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

SEED = 556
rng = np.random.default_rng(SEED)
print("Seed used:", SEED)

ModuleNotFoundError: No module named 'numpy'

## Part 1 — Monte Carlo probability (8 pts)

**Question.** Let $X \sim \text{Binomial}(n=30,\ p=0.25)$. Estimate the event probability $P(X \ge 12)$ by simulation.

**Method.** Draw $N = 10{,}000$ independent Binomial values with `rng.binomial` and take the proportion that satisfy $X \ge 15$. The theoretical value is $P(X\ge 12)=1-F(11)=\sum_{x=12}^{30}\binom{30}{x}0.25^{x}0.75^{30-x}$, computed with `scipy.stats.binom.sf`.

In [ ]:
n, p, k = 30, 0.25, 12
N = 10_000

x = rng.binomial(n=n, p=p, size=N)
p_hat = np.mean(x >= k)

p_true = stats.binom.sf(k - 1, n, p)

mc_se = np.sqrt(p_hat * (1 - p_hat) / N)
ci = (p_hat - 1.96 * mc_se, p_hat + 1.96 * mc_se)

print(f"Monte Carlo estimate : {p_hat:.5f}")
print(f"Theoretical value    : {p_true:.5f}")
print(f"Absolute error       : {abs(p_hat - p_true):.5f}")
print(f"Monte Carlo SE       : {mc_se:.5f}")
print(f"95% MC interval      : ({ci[0]:.5f}, {ci[1]:.5f})")
print(f"Theory inside interval? {ci[0] <= p_true <= ci[1]}")

**Results and Monte Carlo error.** The simulation estimate is about 0.0516, versus the exact value 0.0507 (absolute error ≈ 0.0009). The exact value lies inside the 95% Monte Carlo interval.

Monte Carlo error is the random error that comes from using a finite number of simulations. Each simulated indicator $\mathbf 1\{X_i\ge 12\}$ is Bernoulli($\theta$), so $\hat\theta$ is a sample proportion with
$$\text{SE}(\hat\theta)=\sqrt{\hat\theta(1-\hat\theta)/N}\approx 0.0022 .$$
A different seed would give a slightly different estimate, typically within about ±2 SE of the truth. The error is *not* a bias in the method; it shrinks as $N$ grows.

## Part 2 — Monte Carlo stability (6 pts)

Repeat the same estimate for $N = 100, 1{,}000, 10{,}000, 100{,}000, 1{,}000{,}000$. To show the spread (not just one lucky or unlucky run), each size up to 100,000 is also repeated 200 times and the standard deviation of those 200 estimates is reported.

In [ ]:
sizes = [100, 1_000, 10_000, 100_000, 1_000_000]
reps = 200

rows = []
for N_s in sizes:
    single = np.mean(rng.binomial(n, p, size=N_s) >= k)
    many = np.mean(rng.binomial(n, p, size=(reps, N_s)) >= k, axis=1) if N_s <= 100_000 else None
    rows.append({
        "N": N_s,
        "estimate": single,
        "abs_error": abs(single - p_true),
        "theoretical_MC_SE": np.sqrt(p_true * (1 - p_true) / N_s),
        "empirical_SD_over_200_runs": many.std(ddof=1) if many is not None else np.nan,
    })

stab = pd.DataFrame(rows)
print(f"True value = {p_true:.5f}")
display(stab.round(5)) if "display" in globals() else print(stab.round(5).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogx(stab["N"], stab["estimate"], "o-", label="MC estimate")
axes[0].axhline(p_true, color="red", ls="--", label=f"true = {p_true:.4f}")
axes[0].set_xlabel("Number of simulations N"); axes[0].set_ylabel("Estimated probability")
axes[0].set_title("Estimate stabilizes as N grows"); axes[0].legend()

axes[1].loglog(stab["N"], stab["theoretical_MC_SE"], "s-", label="theoretical SE")
sub = stab.dropna()
axes[1].loglog(sub["N"], sub["empirical_SD_over_200_runs"], "o--", label="empirical SD (200 runs)")
axes[1].set_xlabel("Number of simulations N"); axes[1].set_ylabel("Monte Carlo error")
axes[1].set_title("Error shrinks like 1/sqrt(N)"); axes[1].legend()

plt.tight_layout()
Path("outputs").mkdir(exist_ok=True)
plt.savefig("outputs/mc_stability.png", dpi=150)
plt.show()

**Why more simulations help.** The estimates wander widely at $N=100$ (SE ≈ 0.022, a large fraction of the probability itself; the single run gave 0.04) and settle near the true 0.0507 (by $N=100{,}000$ the estimate is within about 0.0002) as $N$ grows. By the law of large numbers, the sample proportion converges to the true probability, and by the CLT its standard error shrinks like $1/\sqrt N$: the empirical SD across repeated runs matches the theoretical SE in the right panel. Each 10× increase in $N$ cuts the error by only about $\sqrt{10}\approx 3.2$×, so gains are real but slow, and rare events (small $\theta$) need large $N$ for a good *relative* error.

## Part 3 — Bootstrap (10 pts)

**Data.** Student's (1908) sleep data: extra hours of sleep for 10 patients under each of two drugs (a small real dataset; it is the same data as R's `sleep`). It is written to and read from a relative path, `data/sleep.csv`. *If your instructor provided a dataset, replace the data cell with `pd.read_csv("data/<file>.csv")`.*

**Statistic.** Difference in group means, $\bar x_{\text{drug2}}-\bar x_{\text{drug1}}$. Each bootstrap replicate resamples *within* each group with replacement, then recomputes the statistic. $B = 10{,}000 \ge 5{,}000$.

In [ ]:
Path("data").mkdir(exist_ok=True)
sleep = pd.DataFrame({
    "group": ["drug1"] * 10 + ["drug2"] * 10,
    "extra": [0.7, -1.6, -0.2, -1.2, -0.1, 3.4, 3.7, 0.8, 0.0, 2.0,
              1.9,  0.8,  1.1,  0.1, -0.1, 4.4, 5.5, 1.6, 4.6, 3.4],
})
sleep.to_csv("data/sleep.csv", index=False)

sleep = pd.read_csv("data/sleep.csv")
g1 = sleep.loc[sleep["group"] == "drug1", "extra"].to_numpy()
g2 = sleep.loc[sleep["group"] == "drug2", "extra"].to_numpy()
print(sleep.groupby("group")["extra"].agg(["count", "mean", "std", "median"]).round(3))

In [ ]:
B = 10_000
obs_diff = g2.mean() - g1.mean()

boot_diffs = np.empty(B)
for b in range(B):
    s1 = rng.choice(g1, size=g1.size, replace=True)
    s2 = rng.choice(g2, size=g2.size, replace=True)
    boot_diffs[b] = s2.mean() - s1.mean()

boot_se = boot_diffs.std(ddof=1)
ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])

print(f"Observed difference in means : {obs_diff:.3f}")
print(f"Bootstrap SE                 : {boot_se:.3f}")
print(f"95% percentile CI            : ({ci_lo:.3f}, {ci_hi:.3f})")

plt.hist(boot_diffs, bins=50, color="steelblue", edgecolor="white")
plt.axvline(obs_diff, color="red", label="observed")
plt.axvline(ci_lo, color="black", ls="--"); plt.axvline(ci_hi, color="black", ls="--", label="95% percentile CI")
plt.xlabel("Bootstrap difference in means"); plt.ylabel("Count"); plt.title("Bootstrap distribution")
plt.legend(); plt.savefig("outputs/bootstrap_dist.png", dpi=150); plt.show()

**Interpretation.** The observed difference is 1.58 extra hours, with a bootstrap standard error of about 0.80. The 95% percentile interval is roughly (0.02, 3.17). Interpreted: the data are consistent with drug 2 giving anywhere from about zero to about 3.2 more hours of sleep than drug 1. The lower endpoint is barely above 0, so the evidence for a difference is borderline: the interval only just excludes 0 at the 95% level, and a different seed could move that endpoint to either side of 0. Note that with only 10 observations per group the bootstrap interval can be slightly too narrow, since small samples are poorly represented by their empirical distribution.

## Part 4 — Permutation test (10 pts)

- **Hypotheses.** $H_0$: group labels are exchangeable (no drug difference in the distribution of extra sleep) vs. $H_A$: the groups differ (two-sided).
- **Test statistic.** $T=\bar x_{\text{drug2}}-\bar x_{\text{drug1}}$; the two-sided p-value uses $|T|$.
- **Null distribution.** Pool all 20 values, randomly reassign 10 to each group, recompute $T$; repeat 10,000 times.
- **Monte Carlo p-value with the +1 correction:**
$$\hat p=\frac{1+\#\{|T^{*}_b|\ge |T_{\text{obs}}|\}}{B+1}.$$

In [ ]:
pooled = np.concatenate([g1, g2])
n1 = g1.size
T_obs = g2.mean() - g1.mean()

n_perm = 10_000
perm_stats = np.empty(n_perm)
for i in range(n_perm):
    shuffled = rng.permutation(pooled)
    perm_stats[i] = shuffled[n1:].mean() - shuffled[:n1].mean()

count = np.sum(np.abs(perm_stats) >= abs(T_obs))
p_value = (1 + count) / (n_perm + 1)

print(f"Observed statistic T_obs : {T_obs:.3f}")
print(f"Permutations >= |T_obs|  : {count} of {n_perm}")
print(f"Monte Carlo p-value      : {p_value:.4f}")
print(f"(Welch t-test p-value for reference: {stats.ttest_ind(g2, g1, equal_var=False).pvalue:.4f})")

plt.hist(perm_stats, bins=50, color="gray", edgecolor="white")
plt.axvline(T_obs, color="red", label="observed"); plt.axvline(-T_obs, color="red", ls="--")
plt.xlabel("Permuted difference in means"); plt.ylabel("Count"); plt.title("Permutation null distribution")
plt.legend(); plt.savefig("outputs/permutation_null.png", dpi=150); plt.show()

**Interpretation.** The observed difference of 1.58 hours is at least as large in absolute value as about 8% of the label-shuffled differences, giving $\hat p\approx 0.084$ (839 of 10,000 permutations). At $\alpha=0.05$ we do not reject $H_0$: if the drugs had the same effect, a difference this large would arise by chance roughly 8% of the time. This agrees with the Welch t-test (p ≈ 0.079). The bootstrap interval, whose lower end sits just above 0, is the borderline case: the two methods answer different questions and both sit right at the edge of the 5% line, so the conclusion is 'suggestive, not conclusive'. The +1 correction guarantees $\hat p>0$ and keeps the Monte Carlo test valid, since the observed arrangement is itself one of the possible permutations.

**Permutation vs. bootstrap distribution.** They answer different questions.
- The **permutation distribution** is built *under $H_0$*: shuffling labels forces the groups to have the same distribution (resampling **without** replacement from the pooled data). It is centered near 0 and is used to judge how unusual $T_{\text{obs}}$ is, so it yields a p-value.
- The **bootstrap distribution** is built *without assuming $H_0$*: it resamples **with** replacement within each group to mimic repeated sampling from the population. It is centered near the observed statistic (1.58) and is used to measure the variability of the estimate, so it yields a standard error and a confidence interval.

In short: permutation = null distribution for testing; bootstrap = sampling distribution for estimation.

## Part 5 — Reproducibility (6 pts)

- **Explicit RNG object:** `rng = np.random.default_rng(SEED)` in the first cell; every random call (`rng.binomial`, `rng.choice`, `rng.permutation`) uses it. No global `np.random.seed`.
- **Seed recorded:** `SEED = 556` (printed above and in the title).
- **Relative paths only:** `data/sleep.csv`, `outputs/*.png`.
- **Top to bottom:** restart the kernel and Run All before submitting.
- **Commit and push** (from the repo terminal): `git add . && git commit -m "STA 556 HW4" && git push`.

In [ ]:
import sys, platform
print("Seed          :", SEED)
print("RNG           :", type(rng).__name__, "/", rng.bit_generator.__class__.__name__)
print("Python        :", platform.python_version())
print("NumPy / pandas:", np.__version__, "/", pd.__version__)
print("Files written (relative paths):", sorted(str(p) for p in Path(".").glob("outputs/*")) + ["data/sleep.csv"])